In [0]:
df_bronze = spark.table("raw.default.bronze_cms_raw")

print(f"Total rows: {df_bronze.count():,}")
print(f"Total columns: {len(df_bronze.columns)}")
print("\nSchema:")
df_bronze.printSchema()

Total rows: 1,999,999
Total columns: 28

Schema:
root
 |-- Rndrng_NPI: long (nullable = true)
 |-- Rndrng_Prvdr_Last_Org_Name: string (nullable = true)
 |-- Rndrng_Prvdr_First_Name: string (nullable = true)
 |-- Rndrng_Prvdr_MI: string (nullable = true)
 |-- Rndrng_Prvdr_Crdntls: string (nullable = true)
 |-- Rndrng_Prvdr_Ent_Cd: string (nullable = true)
 |-- Rndrng_Prvdr_St1: string (nullable = true)
 |-- Rndrng_Prvdr_St2: string (nullable = true)
 |-- Rndrng_Prvdr_City: string (nullable = true)
 |-- Rndrng_Prvdr_State_Abrvtn: string (nullable = true)
 |-- Rndrng_Prvdr_State_FIPS: string (nullable = true)
 |-- Rndrng_Prvdr_Zip5: string (nullable = true)
 |-- Rndrng_Prvdr_RUCA: double (nullable = true)
 |-- Rndrng_Prvdr_RUCA_Desc: string (nullable = true)
 |-- Rndrng_Prvdr_Cntry: string (nullable = true)
 |-- Rndrng_Prvdr_Type: string (nullable = true)
 |-- Rndrng_Prvdr_Mdcr_Prtcptg_Ind: string (nullable = true)
 |-- HCPCS_Cd: string (nullable = true)
 |-- HCPCS_Desc: string (nullable 

In [0]:
# Cell 2 — data quality checks before cleaning
from pyspark.sql import functions as F

print("=== DATA QUALITY REPORT ===")

print(f"\n1. Rows with null NPI: {df_bronze.filter(F.col('Rndrng_NPI').isNull()).count():,}")
print(f"2. Rows with null HCPCS_Cd: {df_bronze.filter(F.col('HCPCS_Cd').isNull()).count():,}")
print(f"3. Rows with Avg_Sbmtd_Chrg < 0.01: {df_bronze.filter(F.col('Avg_Sbmtd_Chrg') < 0.01).count():,}")
print(f"4. Rows with Avg_Mdcr_Pymt_Amt == 0: {df_bronze.filter(F.col('Avg_Mdcr_Pymt_Amt') == 0).count():,}")
print(f"5. Duplicate NPI+HCPCS+Place rows: {df_bronze.count() - df_bronze.dropDuplicates(['Rndrng_NPI','HCPCS_Cd','Place_Of_Srvc']).count():,}")

=== DATA QUALITY REPORT ===

1. Rows with null NPI: 0
2. Rows with null HCPCS_Cd: 0
3. Rows with Avg_Sbmtd_Chrg < 0.01: 7
4. Rows with Avg_Mdcr_Pymt_Amt == 0: 20
5. Duplicate NPI+HCPCS+Place rows: 0


In [0]:
# Cell 3 — clean, rename columns, add audit columns
from pyspark.sql import functions as F

df_silver = df_bronze \
    .filter(F.col('Avg_Sbmtd_Chrg') >= 0.01) \
    .filter(F.col('Rndrng_NPI').isNotNull()) \
    .select(
        F.col('Rndrng_NPI').alias('provider_id'),
        F.col('Rndrng_Prvdr_Last_Org_Name').alias('provider_last_name'),
        F.col('Rndrng_Prvdr_First_Name').alias('provider_first_name'),
        F.col('Rndrng_Prvdr_MI').alias('provider_middle_initial'),
        F.col('Rndrng_Prvdr_Crdntls').alias('credentials'),
        F.col('Rndrng_Prvdr_Ent_Cd').alias('entity_code'),
        F.col('Rndrng_Prvdr_St1').alias('address_line1'),
        F.col('Rndrng_Prvdr_St2').alias('address_line2'),
        F.col('Rndrng_Prvdr_City').alias('city'),
        F.upper(F.col('Rndrng_Prvdr_State_Abrvtn')).alias('state'),
        F.col('Rndrng_Prvdr_State_FIPS').alias('state_fips'),
        F.col('Rndrng_Prvdr_Zip5').alias('zip_code'),
        F.col('Rndrng_Prvdr_RUCA').alias('ruca_code'),
        F.col('Rndrng_Prvdr_RUCA_Desc').alias('ruca_desc'),
        F.col('Rndrng_Prvdr_Cntry').alias('country'),
        F.col('Rndrng_Prvdr_Type').alias('specialty'),
        F.col('Rndrng_Prvdr_Mdcr_Prtcptg_Ind').alias('medicare_participation_ind'),
        F.col('HCPCS_Cd').alias('procedure_code'),
        F.col('HCPCS_Desc').alias('procedure_desc'),
        F.col('HCPCS_Drug_Ind').alias('drug_indicator'),
        F.upper(F.col('Place_Of_Srvc')).alias('place_of_service'),
        F.col('Tot_Benes').alias('total_beneficiaries'),
        F.col('Tot_Srvcs').alias('total_services'),
        F.col('Tot_Bene_Day_Srvcs').alias('total_beneficiary_days'),
        F.col('Avg_Sbmtd_Chrg').alias('avg_submitted_charge'),
        F.col('Avg_Mdcr_Alowd_Amt').alias('avg_medicare_allowed'),
        F.col('Avg_Mdcr_Pymt_Amt').alias('avg_medicare_paid'),
        F.col('Avg_Mdcr_Stdzd_Amt').alias('avg_medicare_standardized'),
        F.current_timestamp().alias('silver_processed_at'),
        F.lit('cms_part_aa').alias('source_file')
    )

print(f"Bronze rows: {df_bronze.count():,}")
print(f"Silver rows: {df_silver.count():,}")
print(f"Rows dropped: {df_bronze.count() - df_silver.count():,}")
print(f"Columns: {len(df_silver.columns)}")
print("\nNew column names:")
print(df_silver.columns)

Bronze rows: 1,999,999
Silver rows: 1,999,992
Rows dropped: 7
Columns: 30

New column names:
['provider_id', 'provider_last_name', 'provider_first_name', 'provider_middle_initial', 'credentials', 'entity_code', 'address_line1', 'address_line2', 'city', 'state', 'state_fips', 'zip_code', 'ruca_code', 'ruca_desc', 'country', 'specialty', 'medicare_participation_ind', 'procedure_code', 'procedure_desc', 'drug_indicator', 'place_of_service', 'total_beneficiaries', 'total_services', 'total_beneficiary_days', 'avg_submitted_charge', 'avg_medicare_allowed', 'avg_medicare_paid', 'avg_medicare_standardized', 'silver_processed_at', 'source_file']


In [0]:
# Cell 4 — write clean Silver table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("raw.silver.cms_claims_clean")

print(f"Silver table written: {df_silver.count():,} rows")
print(f"Columns: {len(df_silver.columns)}")
print("\nSample:")
df_silver.select("provider_id", "specialty", "procedure_code", 
                  "avg_submitted_charge", "avg_medicare_paid",
                  "silver_processed_at", "source_file").show(3, truncate=False)

Silver table written: 1,999,992 rows
Columns: 30

Sample:
+-----------+----------------+--------------+--------------------+-----------------+--------------------------+-----------+
|provider_id|specialty       |procedure_code|avg_submitted_charge|avg_medicare_paid|silver_processed_at       |source_file|
+-----------+----------------+--------------+--------------------+-----------------+--------------------------+-----------+
|1053340802 |Medical Oncology|J1200         |2.1671003717        |0.7905204461     |2026-04-16 20:08:42.787496|cms_part_aa|
|1053340802 |Medical Oncology|J1439         |3.96                |0.8877880519     |2026-04-16 20:08:42.787496|cms_part_aa|
|1053340802 |Medical Oncology|J1454         |2434.45             |366.15836957     |2026-04-16 20:08:42.787496|cms_part_aa|
+-----------+----------------+--------------+--------------------+-----------------+--------------------------+-----------+
only showing top 3 rows


In [0]:
# Cell 5 — verify Silver table
df_check = spark.table("raw.silver.cms_claims_clean")

print(f"Rows: {df_check.count():,}")
print(f"Columns: {len(df_check.columns)}")
print("\nSchema:")
df_check.printSchema()

Rows: 1,999,992
Columns: 30

Schema:
root
 |-- provider_id: long (nullable = true)
 |-- provider_last_name: string (nullable = true)
 |-- provider_first_name: string (nullable = true)
 |-- provider_middle_initial: string (nullable = true)
 |-- credentials: string (nullable = true)
 |-- entity_code: string (nullable = true)
 |-- address_line1: string (nullable = true)
 |-- address_line2: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- state_fips: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- ruca_code: double (nullable = true)
 |-- ruca_desc: string (nullable = true)
 |-- country: string (nullable = true)
 |-- specialty: string (nullable = true)
 |-- medicare_participation_ind: string (nullable = true)
 |-- procedure_code: string (nullable = true)
 |-- procedure_desc: string (nullable = true)
 |-- drug_indicator: string (nullable = true)
 |-- place_of_service: string (nullable = true)
 |-- total_beneficiaries